In [2]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import files
uploaded = files.upload()

Saving data.csv to data.csv
Saving test.csv to test.csv


In [4]:
test_data = pd.read_csv('test.csv')
train_data = pd.read_csv('data.csv')

In [7]:
train_data['rating'].isnull().sum()

np.int64(0)

In [8]:
all_data = pd.concat([train_data, test_data])
n_users = all_data['user_id'].max()
n_items = all_data['item_id'].max()

print(f"\nتعداد کل کاربران: {n_users}")
print(f"تعداد کل فیلم‌ها: {n_items}")


تعداد کل کاربران: 943
تعداد کل فیلم‌ها: 1682


In [9]:
k = 10
alpha = 0.005
beta = 0.02
epochs = 50

print(f"تعداد ویژگی‌های پنهان (k): {k}")
print(f"نرخ یادگیری (alpha): {alpha}")
print(f"ضریب تنظیم (beta): {beta}")
print(f"تعداد تکرارها (epochs): {epochs}")

P = np.random.normal(loc=0, scale=0.1, size=(n_users + 1, k))
Q = np.random.normal(loc=0, scale=0.1, size=(n_items + 1, k))

print(f"\nشکل ماتریس کاربران (P): {P.shape}")
print(f"شکل ماتریس فیلم‌ها (Q): {Q.shape}")

تعداد ویژگی‌های پنهان (k): 10
نرخ یادگیری (alpha): 0.005
ضریب تنظیم (beta): 0.02
تعداد تکرارها (epochs): 50

شکل ماتریس کاربران (P): (944, 10)
شکل ماتریس فیلم‌ها (Q): (1683, 10)


In [10]:
training_history = []

print("شروع فرآیند آموزش مدل...")

for epoch in range(epochs):
  for user_id, item_id, rating, title in train_data.itertuples(index=False):
    user_vector = P[user_id]
    item_vector = Q[item_id]
    prediction = np.dot(user_vector, item_vector)

    error = rating - prediction

    P[user_id] += alpha * (error * item_vector - beta * user_vector)
    Q[item_id] += alpha * (error * user_vector - beta * item_vector)

  total_squared_error = 0

  for user_id, item_id, rating, title in train_data.itertuples(index=False):
    prediction = P[user_id] @ Q[item_id].T # ضرب داخلی
    total_squared_error += (rating - prediction)**2

  train_rmse = np.sqrt(total_squared_error / len(train_data))
  training_history.append(train_rmse)
  print(f"Epoch {epoch+1:02d}/{epochs}  |  Training RMSE: {train_rmse:.4f}")



print("\nفرآیند آموزش با موفقیت به پایان رسید!")




شروع فرآیند آموزش مدل...
Epoch 01/50  |  Training RMSE: 3.6868
Epoch 02/50  |  Training RMSE: 2.6918
Epoch 03/50  |  Training RMSE: 1.5571
Epoch 04/50  |  Training RMSE: 1.2271
Epoch 05/50  |  Training RMSE: 1.0933
Epoch 06/50  |  Training RMSE: 1.0274
Epoch 07/50  |  Training RMSE: 0.9898
Epoch 08/50  |  Training RMSE: 0.9658
Epoch 09/50  |  Training RMSE: 0.9487
Epoch 10/50  |  Training RMSE: 0.9354
Epoch 11/50  |  Training RMSE: 0.9244
Epoch 12/50  |  Training RMSE: 0.9148
Epoch 13/50  |  Training RMSE: 0.9062
Epoch 14/50  |  Training RMSE: 0.8982
Epoch 15/50  |  Training RMSE: 0.8908
Epoch 16/50  |  Training RMSE: 0.8838
Epoch 17/50  |  Training RMSE: 0.8773
Epoch 18/50  |  Training RMSE: 0.8710
Epoch 19/50  |  Training RMSE: 0.8649
Epoch 20/50  |  Training RMSE: 0.8590
Epoch 21/50  |  Training RMSE: 0.8533
Epoch 22/50  |  Training RMSE: 0.8478
Epoch 23/50  |  Training RMSE: 0.8424
Epoch 24/50  |  Training RMSE: 0.8371
Epoch 25/50  |  Training RMSE: 0.8319
Epoch 26/50  |  Training 

In [12]:
print("\nشروع ارزیابی مدل روی داده‌های آزمون...")
test_squared_error = 0


for user_id, item_id, rating, title in test_data.itertuples(index=False):
  user_vector = P[user_id]
  item_vector = Q[item_id]

  prediction = np.dot(user_vector, item_vector)

  prediction = np.clip(prediction, 1, 5)

  test_squared_error += (rating - prediction)**2

test_mse = test_squared_error / len(test_data)

test_rmse = np.sqrt(test_mse)

print(f"\nRMSE نهایی روی داده‌های آزمون: {test_rmse:.4f}")

if test_rmse < 0.94:
    print("تبریک! شما به هدف پروژه (RMSE < 0.94) رسیدید.")
else:
    print("مقدار RMSE هنوز بالاتر از هدف پروژه است. سعی کنید هایپرپارامترها (k, alpha, beta, epochs) را تغییر دهید.")


شروع ارزیابی مدل روی داده‌های آزمون...

RMSE نهایی روی داده‌های آزمون: 0.9372
تبریک! شما به هدف پروژه (RMSE < 0.94) رسیدید.
